In [7]:
import os
import pandas as pd
import numpy as np
import re

In [27]:
def sort_key(filename):
    match = re.search(r'keypoints_frame_(\d+)\.csv', filename)
    if match:
        return int(match.group(1))
    return filename


def load_csv_files(directory):
    csv_files = sorted([f for f in os.listdir(directory) if f.endswith('.csv')], key=sort_key)
    data_frames = [pd.read_csv(os.path.join(directory, f)) for f in csv_files]
    return data_frames
    
    
def interpolate_frames(data_frames, num_frames=30):
    original_frames = len(data_frames)
    interpolated_frames = []
    
    for i in range(num_frames):
        alpha = i * (original_frames - 1) / (num_frames - 1)
        lower_index = int(np.floor(alpha))
        upper_index = int(np.ceil(alpha))
        
        if lower_index == upper_index:
            interpolated_frames.append(data_frames[lower_index])
        else:
            lower_frame = data_frames[lower_index].copy()
            upper_frame = data_frames[upper_index]
            
            weight = alpha - lower_index
            
            for column in ['x', 'y']:
                lower_values = lower_frame[column].values
                upper_values = upper_frame[column].values
                
                # Handle 0 values by replacing them with NaN
                lower_values = np.where(lower_values == 0, np.nan, lower_values)
                upper_values = np.where(upper_values == 0, np.nan, upper_values)
                
                # Interpolate the values, skipping NaNs
                interpolated_values = np.where(
                    np.isnan(lower_values) & np.isnan(upper_values),
                    0,  # If both are NaN, set the result to 0
                    np.where(
                        np.isnan(lower_values),
                        upper_values,  # If lower is NaN, take upper
                        np.where(
                            np.isnan(upper_values),
                            lower_values,  # If upper is NaN, take lower
                            lower_values * (1 - weight) + upper_values * weight  # Both are present
                        )
                    )
                )
                
                lower_frame[column] = interpolated_values
            
            # Interpolating confidence values
            lower_confidence = lower_frame['z'].values
            upper_confidence = upper_frame['z'].values
            
            interpolated_confidence = np.where(
                np.isnan(lower_confidence) & np.isnan(upper_confidence),
                0,  # If both are NaN, set the result to 0
                np.where(
                    np.isnan(lower_confidence),
                    upper_confidence,  # If lower is NaN, take upper
                    np.where(
                        np.isnan(upper_confidence),
                        lower_confidence,  # If upper is NaN, take lower
                        lower_confidence * (1 - weight) + upper_confidence * weight  # Both are present
                    )
                )
            )
            
            lower_frame['z'] = interpolated_confidence
            
            interpolated_frames.append(lower_frame)
    
    return interpolated_frames


def save_csv_files(data_frames, output_directory):
    if not os.path.exists(output_directory):
        os.makedirs(output_directory)
    for i, df in enumerate(data_frames):
        df.to_csv(os.path.join(output_directory, f'frame_{i:03d}.csv'), index=False)



In [29]:
# Répertoire contenant les fichiers CSV originaux
input_directory = 'Dataset/output_2'
# Répertoire où les nouveaux fichiers CSV seront sauvegardés
output_directory = 'Dataset/output_2_30f'

for pen in os.listdir(input_directory) :
    if pen.startswith('penalty_'):
        print(pen)
        chemin_complet = os.path.join(input_directory, pen)
        chemin_sortie = os.path.join(output_directory, pen)
        # Charger les fichiers CSV
        data_frames = load_csv_files(chemin_complet)
        # Interpoler les frames pour obtenir 30 fichiers
        interpolated_frames = interpolate_frames(data_frames, num_frames=30)
        # Sauvegarder les nouveaux fichiers CSV
        save_csv_files(interpolated_frames, chemin_sortie)



penalty_4
penalty_3
penalty_2
penalty_5
penalty_35
penalty_68
penalty_112
penalty_115
penalty_50
penalty_123
penalty_59
penalty_92
penalty_66
penalty_61
penalty_95
penalty_33
penalty_34
penalty_94
penalty_60
penalty_58
penalty_122
penalty_67
penalty_93
penalty_114
penalty_51
penalty_113
penalty_69
penalty_56
penalty_80
penalty_74
penalty_73
penalty_87
penalty_109
penalty_45
penalty_100
penalty_42
penalty_89
penalty_107
penalty_29
penalty_16
penalty_11
penalty_18
penalty_27
penalty_20
penalty_43
penalty_106
penalty_88
penalty_44
penalty_101
penalty_108
penalty_86
penalty_72
penalty_75
penalty_81
penalty_21
penalty_19
penalty_26
penalty_10
penalty_28
penalty_17
penalty_7
penalty_9
penalty_8
penalty_6
penalty_1
penalty_98
penalty_116
penalty_53
penalty_111
penalty_54
penalty_62
penalty_96
penalty_118
penalty_120
penalty_91
penalty_65
penalty_31
penalty_36
penalty_38
penalty_121
penalty_64
penalty_90
penalty_119
penalty_97
penalty_63
penalty_110
penalty_55
penalty_117
penalty_99
penalty_52